### 01 - Instalação (Bibliotecas)

In [ ]:
%pip install numpy
%pip install matplotlib
%pip install scikit-learn
%pip install gensim

### 02 - Recursos (Importações)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

from gensim.models import Word2Vec, KeyedVectors
import gensim.downloader as api

plt.style.use("ggplot")

### 03 - Implementação Inicial

In [ ]:
sentences = [
  ["gato", "come", "peixe"],
  ["cachorro", "come", "carne"],
  ["gato", "e", "cachorro", "sao", "animais"],
  ["peixe","vive", "na", "agua"],
  ["cachorro", "late"],
  ["gato","mia"],
  ["leao", "come", "carne"],
  ["leao", "e", "um", "animal", "selvagem"],
  ["peixe", "nada", "na", "agua"]
]

model = Word2Vec(
  sentences=sentences,
  vector_size=20,
  window=2,
  min_count=1,
  sg=0,
  epochs=200
)

print(f"Vocabulário:\n\n{list(model.wv.index_to_key)}")

### 04 - Extração de Vetores

In [ ]:
cat_vector = model.wv["gato"]

print(f"Vetor (Gato):\n\n{cat_vector}")
print(f"\nDimensão do Vetor (Gato): {cat_vector.shape}")

### 05 - Similaridade de Tokens

In [ ]:
print(f"Similaridade de Tokens (Gato):\n\n{model.wv.most_similar('gato')}")

### 06 - Similaridade entre Tokens

In [ ]:
cat_dog_similarity = model.wv.similarity("gato", "cachorro")

print(f"Similaridade (Gato/Cachorro): {cat_dog_similarity:.4f}")

cat_fish_similarity = model.wv.similarity("gato", "peixe")

print(f"\nSimilaridade (Gato/Peixe): {cat_fish_similarity:.4f}")

print("\nSimilaridade Geral (Come):")

for (word, score) in model.wv.most_similar("come", topn=3):
  print(f"\n- {word} ({score:.4f})")

print(f"\nPalavra Não Mapeada (Intrusa):\n\n- {model.wv.doesnt_match(['gato', 'cachorro', 'carne'])}")

### 07 - Álgebra Linear

In [ ]:
result = model.wv.most_similar(positive=["carne", "peixe"], negative=["cachorro"], topn=5)

for (word, score) in result:
  print(f"◼ Carne - Cachorro + Peixe = {word:10} ({score:.2f})")

### 08 - Exportando e Importando Dados

In [ ]:
# Salvando os vetores.

model.wv.save("vectors.kv")

vectors_loaded = KeyedVectors.load("vectors.kv")

print(f"Vetores recuperados do disco. Saída:\n\n{vectors_loaded}")

# Salvando o modelo completo.

model.save("vectors.model")

### 09 - Modelos Prontos

In [ ]:
print("Modelos Disponíveis (Download):")

info = api.info()

for model_name in list(info["models"].keys())[:5]:
  print(f"\n- {model_name}")

print("\nBaixando/Carregando o modelo 'glove-twitter-25':")

glove_twitter_model = api.load("glove-twitter-25")

print("\nSimilaridade entre 'Computer' and 'Programming' no GloVe:")
print(f"\n{glove_twitter_model.similarity('computer', 'programming')}")

print("\nAnalogia no GloVe:")

for (word, score) in glove_twitter_model.most_similar(positive=["king", "woman"], negative=["man"], topn=3):
  print(f"\n- {word} ({score:.4f})")

### 10 - Gráficos (PCA)

In [ ]:
words = [
  'car', 'bus', 'truck', 'bicycle', 'train',
  'apple', 'banana', 'orange', 'grape',
  'dog', 'cat', 'bird', 'fish', 'horse',
  'computer', 'software', 'hardware', 'code'
]

vectors = np.array([glove_twitter_model[word] for word in words])

pca = PCA(n_components=2)

vectors_2d = pca.fit_transform(vectors)

plt.figure(figsize=(10, 8))

plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], edgecolors="k", c="c", s=100)

for (i, word) in enumerate(words):
  plt.annotate(
    word,
    xy=(vectors_2d[i, 0], vectors_2d[i, 1]),
    xytext=(5, 2),
    textcoords="offset points",
    ha="right",
    va="bottom"
  )

plt.title("Projeção 2D de Word Embeddings via PCA")

plt.grid(True)

plt.show()